In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import r2_score

In [5]:
df = pd.read_csv("../data/Assets+indicator.csv", index_col = 0, parse_dates=True)

assets = ['기술주', '경기방어주','경기민감주','금','부동산','비트코인']
indicator = [col for col in df.columns if col not in assets]

results = {}
for asset in assets:
    model_df = df[indicator + [asset]].copy()
    
    # ★★★ 핵심: 예측을 위해 자산(Y)을 한 달 당깁니다 (Next Month Prediction) ★★★
    model_df[asset] = model_df[asset].shift(-1)
    
    # Shift로 인해 마지막 달은 정답이 없으므로(NaN) 제거
    model_df = model_df.dropna()

    if len(model_df) < 20: # 데이터가 너무 적으면 패스
        print(f"데이터 부족으로 건너뜁니다.")
        continue

    # X, y 분리
    X = model_df[indicator]
    y = model_df[asset]

    split_idx = int(len(model_df) * 0.8)
    
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    

    # 모델 훈련
    lgbm = lgb.LGBMRegressor(random_state=42, n_estimators=100, verbose=-1)
    lgbm.fit(X_train, y_train)
    
    # 예측 및 평가
    pred = lgbm.predict(X_test)
    r2 = r2_score(y_test, pred)
    results[asset] = r2
    
    print(f"R² Score: {r2:.4f}")

# --- 3. 최종 결과 ---
print("\n=== 최종 R² 성적표 (월별 예측) ===")
print(pd.Series(results).sort_values(ascending=False))


R² Score: -0.4114
R² Score: -0.2810
R² Score: -0.1648
R² Score: -0.5529
R² Score: -0.3104
R² Score: -0.8781

=== 최종 R² 성적표 (월별 예측) ===
경기민감주   -0.164837
경기방어주   -0.280950
부동산     -0.310427
기술주     -0.411430
금       -0.552871
비트코인    -0.878139
dtype: float64
